# Assignment 2 - Graph Analysis of Architectural Structure

This notebook builds on the Assignment 1 workflow, but uses the Aranjuez model files selected for Assignment 2:

- `Aranjuez_Test_05_Rooms.obj`
- `Aranjuez_Test_02_Doors.obj`
- `Aranjuez_Test_02_Windows.obj`

The analysis uses TopologicPy graph methods to compute centrality, shortest paths, bottlenecks, communities, and spatial organization summaries.


## 1. Import libraries and check TopologicPy version


In [ ]:
import csv
import math
import re
from collections import defaultdict, deque
from itertools import combinations
from pathlib import Path

from IPython.display import Markdown, display

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Color import Color
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Grid import Grid
from topologicpy.Helper import Helper
from topologicpy.Graph import Graph
from topologicpy.Shell import Shell
from topologicpy.Wire import Wire

print("This tutorial requires topologicpy version 0.9.33 or newer.")
version_text = str(Helper.Version())
print(version_text)

def version_tuple(text):
    numbers = re.findall(r"\d+", str(text))
    return tuple(int(n) for n in numbers[:3]) if numbers else (0, 0, 0)

print("This tutorial requires topologicpy version 0.9.33 or newer.")
print(Helper.Version())


## 2. Set paths and renderer


In [ ]:
PROJECT_DIR = Path(r"C:\Users\Arq. David Agudelo\OneDrive\Documentos\GitHub\GRAPHML_DAVID_AGUDELO_2026")
ROOMS_OBJ = PROJECT_DIR / "Aranjuez_Test_05_Rooms.obj"
DOORS_OBJ = PROJECT_DIR / "Aranjuez_Test_02_Doors.obj"
WINDOWS_OBJ = PROJECT_DIR / "Aranjuez_Test_02_Windows.obj"
OUTPUT_DIR = PROJECT_DIR / "Assignment_02_Outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

renderer = "vscode"
CONNECTION_TOLERANCES = [0.08, 0.15, 0.30, 0.60, 1.00]

for path in [ROOMS_OBJ, DOORS_OBJ, WINDOWS_OBJ]:
    if not path.exists():
        raise FileNotFoundError(path)
    print("Found:", path.name)


## 3. Helper functions


In [ ]:
def get_dict_value(topology, key, default=None):
    d = Topology.Dictionary(topology)
    if not d:
        return default
    value = Dictionary.ValueAtKey(d, key)
    return default if value is None else value


def set_dictionary(topology, values):
    return Topology.SetDictionary(
        topology,
        Dictionary.ByKeysValues(list(values.keys()), list(values.values()))
    )


def get_centroid_xyz(topology, decimals=None):
    c = Topology.Centroid(topology)
    xyz = (float(Vertex.X(c)), float(Vertex.Y(c)), float(Vertex.Z(c)))
    if decimals is None:
        return xyz
    return tuple(round(v, decimals) for v in xyz)


def get_bbox(topology):
    vertices = Topology.Vertices(topology)
    if not vertices:
        return None
    xs = [float(Vertex.X(v)) for v in vertices]
    ys = [float(Vertex.Y(v)) for v in vertices]
    zs = [float(Vertex.Z(v)) for v in vertices]
    return [min(xs), min(ys), min(zs), max(xs), max(ys), max(zs)]


def point_inside_bbox(point, bbox, tolerance=0.08):
    if not bbox or len(bbox) < 6:
        return False
    x, y, z = point
    min_x, min_y, min_z, max_x, max_y, max_z = bbox[:6]
    return (
        min_x - tolerance <= x <= max_x + tolerance and
        min_y - tolerance <= y <= max_y + tolerance and
        min_z - tolerance <= z <= max_z + tolerance
    )


def get_face_dimensions(face, decimals=3):
    vertices = Topology.Vertices(face)
    xs = [float(Vertex.X(v)) for v in vertices]
    ys = [float(Vertex.Y(v)) for v in vertices]
    zs = [float(Vertex.Z(v)) for v in vertices]
    dims = sorted([max(xs)-min(xs), max(ys)-min(ys), max(zs)-min(zs)], reverse=True)
    return tuple(round(v, decimals) for v in dims)


def vertex_coord_key(v, decimals=6):
    return (
        round(float(Vertex.X(v)), decimals),
        round(float(Vertex.Y(v)), decimals),
        round(float(Vertex.Z(v)), decimals)
    )


def edge_length(edge):
    vertices = Topology.Vertices(edge)
    if len(vertices) < 2:
        return 0.0
    a = get_centroid_xyz(vertices[0])
    b = get_centroid_xyz(vertices[1])
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2)


def write_csv(path, rows):
    path = Path(path)
    if not rows:
        print("No rows to export:", path.name)
        return
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print("Exported:", path)


def print_rows(rows, limit=12):
    if not rows:
        print("No rows")
        return
    sample = rows[:limit]
    columns = list(sample[0].keys())
    widths = {c: max(len(str(c)), *(len(str(r.get(c, ""))) for r in sample)) for c in columns}
    print(" | ".join(str(c).ljust(widths[c]) for c in columns))
    print("-+-".join("-" * widths[c] for c in columns))
    for row in sample:
        print(" | ".join(str(row.get(c, "")).ljust(widths[c]) for c in columns))
    if len(rows) > limit:
        print(f"... {len(rows) - limit} more rows")


def color_for_room(name):
    name_l = str(name).lower()
    if "circulation" in name_l or "lobby" in name_l or "hall" in name_l:
        return "gold"
    if "stair" in name_l or "elevator" in name_l:
        return "purple"
    if "storage" in name_l or "technical" in name_l or "recycling" in name_l:
        return "lightgray"
    if "bath" in name_l or "toilet" in name_l:
        return "lightblue"
    return "lightgreen"


## 4. Load Aranjuez rooms and reconstruct the spatial model


In [ ]:
room_objects = Topology.ByOBJPath(str(ROOMS_OBJ))
print("Room objects imported:", len(room_objects))

cells = []
selectors = []

for i, obj in enumerate(room_objects, start=1):
    faces = Topology.Faces(obj)
    if len(faces) <= 1:
        continue

    try:
        cell = Cell.ByFaces(faces)
    except Exception as error:
        print("Skipped room object that could not become a cell:", i, error)
        continue

    if cell is None:
        print("Skipped empty cell from object:", i)
        continue

    try:
        cell = Topology.RemoveCollinearEdges(cell)
    except Exception:
        pass

    obj_name = get_dict_value(obj, "name", f"Room_{i:03d}")
    try:
        selector = Topology.InternalVertex(cell)
    except Exception:
        selector = Topology.Centroid(cell)
    if selector is None:
        selector = Topology.Centroid(cell)

    bbox = get_bbox(cell)
    cx, cy, cz = get_centroid_xyz(cell, decimals=6)

    cell = set_dictionary(cell, {
        "type": "room",
        "name": obj_name,
        "label": obj_name,
        "source_id": f"R{i:03d}",
        "color": color_for_room(obj_name),
        "vertex_size": 18,
        "aabb": bbox,
        "center_x": cx,
        "center_y": cy,
        "center_z": cz,
    })

    selector = set_dictionary(selector, {
        "type": "room",
        "name": obj_name,
        "label": obj_name,
        "source_id": f"R{i:03d}",
        "color": color_for_room(obj_name),
        "vertex_size": 18,
        "aabb": bbox,
    })

    cells.append(cell)
    selectors.append(selector)

house = CellComplex.ByCells(cells)
try:
    house = Topology.TransferDictionariesBySelectors(house, selectors, tranCells=True)
except Exception as error:
    print("Dictionary transfer warning:", error)

house_cells = Topology.Cells(house)

print("Cells in house:", len(house_cells))
for cell in house_cells[:10]:
    print(get_dict_value(cell, "name"), get_centroid_xyz(cell, decimals=3))


## 5. Import doors and windows as apertures


In [ ]:
def load_aperture_faces(path, aperture_type, color):
    objects = Topology.ByOBJPath(str(path))
    apertures = []
    for i, obj in enumerate(objects, start=1):
        faces = Topology.Faces(obj)
        if not faces:
            continue
        face = faces[0]
        try:
            face = Topology.RemoveCollinearEdges(face)
        except Exception:
            pass
        obj_name = get_dict_value(obj, "name", f"{aperture_type.title()}_{i:03d}")
        bbox = get_bbox(face)
        cx, cy, cz = get_centroid_xyz(face, decimals=6)
        width, height, thickness = get_face_dimensions(face)
        face = set_dictionary(face, {
            "type": aperture_type,
            "aperture_type": aperture_type,
            "name": obj_name,
            "label": obj_name,
            "source_id": f"{aperture_type[0].upper()}{i:03d}",
            "color": color,
            "vertex_size": 14,
            "aabb": bbox,
            "center_x": cx,
            "center_y": cy,
            "center_z": cz,
            "width": width,
            "height": height,
            "thickness": thickness,
        })
        apertures.append(face)
    return apertures


door_apertures = load_aperture_faces(DOORS_OBJ, "door", "brown")
window_apertures = load_aperture_faces(WINDOWS_OBJ, "window", "cyan")
apertures = door_apertures + window_apertures

print("Doors imported:", len(door_apertures))
print("Windows imported:", len(window_apertures))
print("Total apertures:", len(apertures))

try:
    house_with_apertures = Topology.AddApertures(house, apertures, subTopologyType="face")
except Exception as error:
    print("Topology.AddApertures did not complete; continuing with graph construction from aperture centroids.")
    print(error)
    house_with_apertures = house


## 6. Detect room-aperture connections


In [ ]:
def get_connected_rooms(aperture, cells, tolerances=CONNECTION_TOLERANCES):
    point = get_centroid_xyz(aperture)
    best = []
    best_tol = None

    for tolerance in tolerances:
        connected = []
        for cell in cells:
            bbox = get_dict_value(cell, "aabb", None) or get_bbox(cell)
            if point_inside_bbox(point, bbox, tolerance=tolerance):
                connected.append(get_dict_value(cell, "name", "Unnamed room"))
        if connected:
            best = sorted(set(connected))
            best_tol = tolerance
            if len(best) >= 2 or get_dict_value(aperture, "type", "") == "window":
                break

    return best, best_tol


aperture_connection_rows = []
aperture_connections = {}

for aperture in apertures:
    name = get_dict_value(aperture, "name")
    aperture_type = get_dict_value(aperture, "type")
    rooms, tolerance = get_connected_rooms(aperture, house_cells)
    aperture_connections[name] = rooms
    aperture_connection_rows.append({
        "aperture": name,
        "type": aperture_type,
        "connected_room_count": len(rooms),
        "connected_rooms": " | ".join(rooms),
        "tolerance_used": tolerance,
    })

write_csv(OUTPUT_DIR / "A2_aperture_connections.csv", aperture_connection_rows)
print_rows(aperture_connection_rows, limit=18)

unconnected = [r for r in aperture_connection_rows if r["connected_room_count"] == 0]
print("Unconnected apertures:", len(unconnected))


## 7. Build room adjacency graph and room-aperture access graph


In [ ]:
ROOM_COLOR = "lightgray"
DOOR_COLOR = "saddlebrown"
WINDOW_COLOR = "deepskyblue"
EDGE_COLOR = "gainsboro"

room_vertices = []
room_vertex_by_name = {}

for i, cell in enumerate(house_cells, start=1):
    room_name = get_dict_value(cell, "name", f"Room_{i:03d}")
    v = Topology.Centroid(cell)
    v = set_dictionary(v, {
        "type": "room",
        "name": room_name,
        "label": room_name,
        "color": ROOM_COLOR,
        "vertex_size": 9,
    })
    room_vertices.append(v)
    room_vertex_by_name[room_name] = v

# Room adjacency graph: rooms become adjacent when they share a door.
room_edges = []
room_pair_to_doors = defaultdict(list)
for door in door_apertures:
    door_name = get_dict_value(door, "name")
    rooms = aperture_connections.get(door_name, [])
    for room_a, room_b in combinations(sorted(set(rooms)), 2):
        key = tuple(sorted([room_a, room_b]))
        room_pair_to_doors[key].append(door_name)

for (room_a, room_b), door_names in room_pair_to_doors.items():
    if room_a not in room_vertex_by_name or room_b not in room_vertex_by_name:
        continue

    edge = Edge.ByVertices([room_vertex_by_name[room_a], room_vertex_by_name[room_b]])
    edge = set_dictionary(edge, {
        "type": "room_adjacency",
        "name": " + ".join(door_names[:3]),
        "doors": " | ".join(door_names),
        "door_count": len(door_names),
        "color": EDGE_COLOR,
        "width": 0.8,
        "weight": round(edge_length(edge), 6),
    })
    room_edges.append(edge)

room_adjacency_graph = Graph.ByVerticesEdges(room_vertices, room_edges)

# Access graph: rooms connect to doors and windows. Exterior apertures become useful exit proxies.
aperture_vertices = []
aperture_vertex_by_name = {}
access_edges = []

for aperture in apertures:
    aperture_name = get_dict_value(aperture, "name")
    aperture_type = get_dict_value(aperture, "type")
    color = DOOR_COLOR if aperture_type == "door" else WINDOW_COLOR
    v = Topology.Centroid(aperture)
    v = set_dictionary(v, {
        "type": aperture_type,
        "name": aperture_name,
        "label": aperture_name,
        "color": color,
        "vertex_size": 7,
    })
    aperture_vertices.append(v)
    aperture_vertex_by_name[aperture_name] = v

    for room_name in aperture_connections.get(aperture_name, []):
        if room_name not in room_vertex_by_name:
            continue
        edge = Edge.ByVertices([room_vertex_by_name[room_name], v])
        edge = set_dictionary(edge, {
            "type": "access",
            "name": f"{room_name}--{aperture_name}",
            "color": EDGE_COLOR,
            "width": 0.6,
            "weight": round(edge_length(edge), 6),
        })
        access_edges.append(edge)

access_graph = Graph.ByVerticesEdges(room_vertices + aperture_vertices, access_edges)
door_space_graph = access_graph

print("Room adjacency graph")
print("Vertices:", len(Graph.Vertices(room_adjacency_graph)))
print("Edges:", len(Graph.Edges(room_adjacency_graph)))
print("Access graph")
print("Vertices:", len(Graph.Vertices(access_graph)))
print("Edges:", len(Graph.Edges(access_graph)))


## 8. Base Graph Preview

Neutral preview of vertices and edges only. Use this only to check that the graph exists; the presentation heatmaps are in Section 15.


In [ ]:
try:
    Graph.Show(
        access_graph,
        vertexSizeKey="vertex_size",
        vertexColorKey="color",
        vertexLabelKey="label",
        showVertexLabel=False,
        edgeWidthKey="width",
        edgeColorKey="color",
        backgroundColor="white",
        renderer=renderer,
    )
except Exception as error:
    print("Base access graph preview unavailable:", error)


In [ ]:
try:
    Graph.Show(
        room_adjacency_graph,
        vertexSizeKey="vertex_size",
        vertexColorKey="color",
        vertexLabelKey="label",
        showVertexLabel=False,
        edgeWidthKey="width",
        edgeColorKey="color",
        backgroundColor="white",
        renderer=renderer,
    )
except Exception as error:
    print("Base room graph preview unavailable:", error)


## 9. Compute graph metrics with TopologicPy


In [ ]:
def compute_graph_metric(graph, metric_name, **kwargs):
    metric = getattr(Graph, metric_name)
    values = metric(graph, **kwargs)
    if values is None:
        return []
    return list(values) if isinstance(values, (list, tuple)) else values


room_vertices_for_metrics = Graph.Vertices(room_adjacency_graph)
access_vertices_for_metrics = Graph.Vertices(access_graph)

room_degree_values = compute_graph_metric(
    room_adjacency_graph,
    "DegreeCentrality",
    normalize=True,
    key="degree_centrality",
)
room_closeness_values = compute_graph_metric(
    room_adjacency_graph,
    "ClosenessCentrality",
    normalize=True,
    key="closeness_centrality",
)
access_betweenness_values = compute_graph_metric(
    access_graph,
    "BetweennessCentrality",
    normalize=True,
    key="betweenness_centrality",
)

try:
    Graph.CommunityPartition(room_adjacency_graph, key="community", colorScale="thermal")
except Exception as error:
    print("CommunityPartition did not complete; trying Graph.Community instead.")
    print(error)
    try:
        Graph.Community(room_adjacency_graph, key="community")
    except Exception as community_error:
        print("Community analysis unavailable:", community_error)

print("TopologicPy graph metrics computed.")
print("Room metric vertices:", len(room_vertices_for_metrics))
print("Access metric vertices:", len(access_vertices_for_metrics))


## 10. Export room centrality metrics


In [ ]:
def graph_name_map(graph):
    vertices = Graph.Vertices(graph)
    by_coord = {}
    by_name = {}
    for i, v in enumerate(vertices, start=1):
        name = get_dict_value(v, "name", f"Node_{i:03d}")
        by_coord[vertex_coord_key(v)] = name
        by_name[name] = v
    return by_coord, by_name


def graph_adjacency(graph):
    by_coord, by_name = graph_name_map(graph)
    adjacency = defaultdict(set)
    edge_rows = []
    for edge in Graph.Edges(graph):
        vertices = Topology.Vertices(edge)
        if len(vertices) < 2:
            continue
        a = by_coord.get(vertex_coord_key(vertices[0]))
        b = by_coord.get(vertex_coord_key(vertices[1]))
        if not a or not b:
            continue
        adjacency[a].add(b)
        adjacency[b].add(a)
        edge_rows.append({
            "node_a": a,
            "node_b": b,
            "weight": round(edge_length(edge), 6),
            "type": get_dict_value(edge, "type", "edge"),
            "name": get_dict_value(edge, "name", ""),
        })
    return adjacency, edge_rows, by_name


def metric_values_by_name(vertices, values, fallback=0):
    result = {}
    if not isinstance(values, list):
        values = []
    for i, vertex in enumerate(vertices):
        name = get_dict_value(vertex, "name", f"Node_{i+1:03d}")
        result[name] = values[i] if i < len(values) else fallback
    return result


room_adjacency, room_edge_rows, room_vertices_by_name = graph_adjacency(room_adjacency_graph)
access_adjacency, access_edge_rows, access_vertices_by_name = graph_adjacency(access_graph)
room_degree_by_name = metric_values_by_name(room_vertices_for_metrics, room_degree_values)
room_closeness_by_name = metric_values_by_name(room_vertices_for_metrics, room_closeness_values)

room_metric_rows = []
for name, vertex in room_vertices_by_name.items():
    room_metric_rows.append({
        "room": name,
        "degree": len(room_adjacency.get(name, [])),
        "degree_centrality": round(float(room_degree_by_name.get(name, 0) or 0), 6),
        "closeness_centrality": round(float(room_closeness_by_name.get(name, 0) or 0), 6),
        "community": get_dict_value(vertex, "community", ""),
    })

room_metric_rows = sorted(
    room_metric_rows,
    key=lambda r: (r["degree_centrality"], r["closeness_centrality"], r["degree"]),
    reverse=True,
)

write_csv(OUTPUT_DIR / "A2_room_graph_metrics.csv", room_metric_rows)
write_csv(OUTPUT_DIR / "A2_room_adjacency_edges.csv", room_edge_rows)
print_rows(room_metric_rows, limit=15)


## 11. Export access graph betweenness metrics


In [ ]:
access_betweenness_by_name = metric_values_by_name(access_vertices_for_metrics, access_betweenness_values)

access_metric_rows = []
for name, vertex in access_vertices_by_name.items():
    access_metric_rows.append({
        "node": name,
        "type": get_dict_value(vertex, "type", "unknown"),
        "degree": len(access_adjacency.get(name, [])),
        "betweenness_centrality": round(float(access_betweenness_by_name.get(name, 0) or 0), 6),
    })

access_metric_rows = sorted(
    access_metric_rows,
    key=lambda r: (r["betweenness_centrality"], r["degree"]),
    reverse=True,
)

write_csv(OUTPUT_DIR / "A2_access_graph_metrics.csv", access_metric_rows)
write_csv(OUTPUT_DIR / "A2_access_graph_edges.csv", access_edge_rows)
print_rows(access_metric_rows, limit=20)


## 12. Shortest paths to exterior aperture proxies


In [ ]:
def weighted_adjacency_from_edges(edge_rows):
    graph = defaultdict(list)
    for row in edge_rows:
        a = row["node_a"]
        b = row["node_b"]
        w = float(row.get("weight", 1) or 1)
        graph[a].append((b, w))
        graph[b].append((a, w))
    return graph


def path_distance(path, adjacency):
    total = 0.0
    for a, b in zip(path[:-1], path[1:]):
        options = [w for neighbor, w in adjacency.get(a, []) if neighbor == b]
        total += min(options) if options else 0.0
    return total


def dijkstra_path(adjacency, start, goal):
    # Standard-library fallback over the TopologicPy graph edges.
    unvisited = set(adjacency.keys()) | {start, goal}
    distances = {node: math.inf for node in unvisited}
    previous = {node: None for node in unvisited}
    distances[start] = 0.0

    while unvisited:
        current = min(unvisited, key=lambda n: distances.get(n, math.inf))
        if current == goal or distances[current] == math.inf:
            break
        unvisited.remove(current)
        for neighbor, weight in adjacency.get(current, []):
            if neighbor not in unvisited:
                continue
            candidate = distances[current] + weight
            if candidate < distances[neighbor]:
                distances[neighbor] = candidate
                previous[neighbor] = current

    if distances.get(goal, math.inf) == math.inf:
        return None, math.inf

    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = previous[current]
    path.reverse()
    return path, distances[goal]


def topologic_shortest_path_names(graph, source_name, target_name, fallback_adjacency):
    source = access_vertices_by_name.get(source_name)
    target = access_vertices_by_name.get(target_name)
    if source and target:
        try:
            path_topology = Graph.ShortestPath(graph, source, target)
            vertices = Topology.Vertices(path_topology)
            if vertices:
                coord_to_name, _ = graph_name_map(graph)
                names = [coord_to_name.get(vertex_coord_key(v), "") for v in vertices]
                names = [n for n in names if n]
                if names:
                    return names, path_distance(names, fallback_adjacency)
        except Exception:
            pass
    return dijkstra_path(fallback_adjacency, source_name, target_name)


access_weighted_adjacency = weighted_adjacency_from_edges(access_edge_rows)
room_nodes = [r["room"] for r in room_metric_rows]

# Exterior proxies: apertures connected to exactly one room. Windows are preferred; exterior doors are retained too.
exit_proxy_rows = [
    row for row in aperture_connection_rows
    if row["connected_room_count"] == 1 and row["aperture"] in access_vertices_by_name
]
exit_proxy_rows = sorted(
    exit_proxy_rows,
    key=lambda r: (0 if r["type"] == "window" else 1, r["aperture"])
)
exit_proxies = [row["aperture"] for row in exit_proxy_rows]

print("Exterior aperture proxies:", len(exit_proxies))
print(exit_proxies[:20])

shortest_path_rows = []
for room_name in room_nodes:
    best = None
    for exit_name in exit_proxies:
        path, distance = topologic_shortest_path_names(access_graph, room_name, exit_name, access_weighted_adjacency)
        if not path or distance == math.inf:
            continue
        row = {
            "start_room": room_name,
            "nearest_exit_proxy": exit_name,
            "path_length_model_units": round(float(distance), 6),
            "hops": max(len(path) - 1, 0),
            "path": " -> ".join(path),
        }
        if best is None or row["path_length_model_units"] < best["path_length_model_units"]:
            best = row
    if best:
        shortest_path_rows.append(best)

shortest_path_rows = sorted(shortest_path_rows, key=lambda r: r["path_length_model_units"], reverse=True)
write_csv(OUTPUT_DIR / "A2_shortest_paths.csv", shortest_path_rows)
print_rows(shortest_path_rows, limit=15)


## 13. Bridges, cut vertices, and bottlenecks


In [ ]:
bottleneck_rows = []

try:
    bridges = Graph.Bridges(access_graph, key="bridge")
except Exception as error:
    print("Graph.Bridges unavailable:", error)
    bridges = []

try:
    cut_vertices = Graph.CutVertices(access_graph, key="cut")
except Exception as error:
    print("Graph.CutVertices unavailable:", error)
    cut_vertices = []

coord_to_access_name, _ = graph_name_map(access_graph)

bridge_rows = []
for i, bridge in enumerate(bridges or [], start=1):
    vertices = Topology.Vertices(bridge)
    if len(vertices) >= 2:
        bridge_rows.append({
            "bridge_id": f"B{i:03d}",
            "node_a": coord_to_access_name.get(vertex_coord_key(vertices[0]), ""),
            "node_b": coord_to_access_name.get(vertex_coord_key(vertices[1]), ""),
        })

cut_vertex_rows = []
for vertex in cut_vertices or []:
    name = coord_to_access_name.get(vertex_coord_key(vertex), get_dict_value(vertex, "name", ""))
    cut_vertex_rows.append({
        "node": name,
        "type": get_dict_value(vertex, "type", "unknown"),
    })

for row in access_metric_rows[:25]:
    if row["betweenness_centrality"] > 0:
        bottleneck_rows.append({
            "node": row["node"],
            "type": row["type"],
            "degree": row["degree"],
            "betweenness_centrality": row["betweenness_centrality"],
            "interpretation": "critical connector / route dependency",
        })

write_csv(OUTPUT_DIR / "A2_bridge_edges.csv", bridge_rows)
write_csv(OUTPUT_DIR / "A2_cut_vertices.csv", cut_vertex_rows)
write_csv(OUTPUT_DIR / "A2_bottlenecks.csv", bottleneck_rows)

print("Bridge edges:")
print_rows(bridge_rows, limit=15)
print("Cut vertices:")
print_rows(cut_vertex_rows, limit=15)
print("Top bottlenecks:")
print_rows(bottleneck_rows, limit=15)


## 14. Global Graph Summary and Exports

This section only writes summary tables and optional graph files. It is not meant to be the main visual reading of the building. Use the heatmap views and the convention below for presentation graphics.


In [ ]:
global_summary_rows = []

for graph_name, graph in [
    ("room_adjacency_graph", room_adjacency_graph),
    ("access_graph", access_graph),
]:
    row = {
        "graph": graph_name,
        "vertices": len(Graph.Vertices(graph)),
        "edges": len(Graph.Edges(graph)),
    }
    for metric_name, output_key in [
        ("Density", "density"),
        ("Diameter", "diameter"),
        ("Order", "order"),
        ("Size", "size"),
    ]:
        try:
            row[output_key] = Graph.__dict__[metric_name](graph)
        except Exception:
            row[output_key] = ""
    global_summary_rows.append(row)

write_csv(OUTPUT_DIR / "A2_global_summary.csv", global_summary_rows)
print_rows(global_summary_rows)

try:
    Graph.ExportToJSON(access_graph, str(OUTPUT_DIR / "A2_access_graph.json"))
    print("Exported A2_access_graph.json")
except Exception as error:
    print("JSON graph export unavailable:", error)

try:
    Graph.ExportToGEXF(access_graph, str(OUTPUT_DIR / "A2_access_graph.gexf"))
    print("Exported A2_access_graph.gexf")
except Exception as error:
    print("GEXF graph export unavailable:", error)

try:
    fig = Graph.AdjacencyMatrixFigure(room_adjacency_graph)
    fig.show()
except Exception as error:
    print("Adjacency matrix figure unavailable:", error)


## 15. Presentation Heatmaps

These are the graphics to use in the presentation/PDF. Each view isolates one reading: connectivity, accessibility, bottlenecks, routes, or clusters. Blue means low value, yellow/orange means medium value, and red means high value.


In [ ]:
HEATMAP_COLORS = [
    "#313695", "#4575b4", "#74add1", "#abd9e9",
    "#ffffbf", "#fdae61", "#f46d43", "#d73027", "#a50026"
]
COMMUNITY_COLORS = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    "#393b79", "#637939", "#8c6d31", "#843c39", "#7b4173"
]
NEUTRAL_EDGE = "#d4d4d4"
HIGHLIGHT_EDGE = "#d73027"
NEUTRAL_NODE = "#d9d9d9"


def metric_float(value, default=0.0):
    try:
        return float(value)
    except Exception:
        return default


def normalize_metric(value, min_value, max_value):
    value = metric_float(value)
    if max_value == min_value:
        return 0.0
    return max(0.0, min(1.0, (value - min_value) / (max_value - min_value)))


def heatmap_color(value, min_value, max_value):
    t = normalize_metric(value, min_value, max_value)
    index = min(len(HEATMAP_COLORS) - 1, int(round(t * (len(HEATMAP_COLORS) - 1))))
    return HEATMAP_COLORS[index]


def metric_range(rows, metric_key):
    values = [metric_float(row.get(metric_key, 0)) for row in rows]
    return (min(values), max(values)) if values else (0, 0)


def compact_label(name):
    text = str(name)
    text = text.replace("Circulation_", "Circ_")
    text = text.replace("Storage_room_", "Stg_")
    text = text.replace("Parking_", "Park_")
    text = text.replace("Apartment_", "Apt_")
    text = text.replace("Room_", "R_")
    return text


def is_circulation_name(name):
    value = str(name).lower()
    return "circulation" in value or "lobby" in value or "hall" in value or "stair" in value


def room_metric_lookup(metric_key):
    low, high = metric_range(room_metric_rows, metric_key)
    lookup = {}
    for row in room_metric_rows:
        value = metric_float(row.get(metric_key, 0))
        lookup[row["room"]] = {
            "value": value,
            "color": heatmap_color(value, low, high),
            "size": round(7 + 30 * normalize_metric(value, low, high), 3),
        }
    return lookup


def access_metric_lookup(metric_key="betweenness_centrality"):
    low, high = metric_range(access_metric_rows, metric_key)
    lookup = {}
    for row in access_metric_rows:
        value = metric_float(row.get(metric_key, 0))
        lookup[row["node"]] = {
            "value": value,
            "color": heatmap_color(value, low, high),
            "size": round(5 + 28 * normalize_metric(value, low, high), 3),
            "type": row.get("type", "unknown"),
        }
    return lookup


def room_betweenness_lookup():
    room_rows = [row for row in access_metric_rows if str(row.get("type", "")).lower() == "room"]
    low, high = metric_range(room_rows, "betweenness_centrality")
    lookup = {}
    for row in room_rows:
        value = metric_float(row.get("betweenness_centrality", 0))
        lookup[row["node"]] = {
            "value": value,
            "color": heatmap_color(value, low, high),
            "size": round(7 + 30 * normalize_metric(value, low, high), 3),
        }
    return lookup


def max_lookup_value(metric_lookup):
    values = [metric_float(item.get("value", 0)) for item in metric_lookup.values()]
    return max(values) if values else 0


def graph_from_metric(base_vertices_by_name, edge_rows, metric_lookup, title, neutral_color=NEUTRAL_NODE, label_policy="none"):
    max_value = max_lookup_value(metric_lookup)
    visual_vertices = {}
    for name, base_vertex in base_vertices_by_name.items():
        info = metric_lookup.get(name, {})
        raw_value = info.get("value", 0)
        value = metric_float(raw_value)
        value_label = round(value, 4) if isinstance(raw_value, (int, float)) else raw_value
        normalized = normalize_metric(value, 0, max_value)

        label = ""
        if label_policy == "all":
            label = f"{compact_label(name)}: {value_label}"
        elif label_policy == "high_or_circulation" and (normalized >= 0.82 or is_circulation_name(name)):
            label = compact_label(name)
        elif label_policy == "high" and normalized >= 0.82:
            label = compact_label(name)

        vertex = Vertex.ByCoordinates(Vertex.X(base_vertex), Vertex.Y(base_vertex), Vertex.Z(base_vertex))
        vertex = set_dictionary(vertex, {
            "name": name,
            "label": label,
            "color": info.get("color", neutral_color),
            "vertex_size": info.get("size", 5),
            "metric_value": round(value, 6),
        })
        visual_vertices[name] = vertex

    visual_edges = []
    for row in edge_rows:
        a = row["node_a"]
        b = row["node_b"]
        if a not in visual_vertices or b not in visual_vertices:
            continue
        edge_value = max(
            metric_float(metric_lookup.get(a, {}).get("value", 0)),
            metric_float(metric_lookup.get(b, {}).get("value", 0)),
        )
        edge = Edge.ByVertices([visual_vertices[a], visual_vertices[b]])
        edge = set_dictionary(edge, {
            "name": row.get("name", ""),
            "color": HIGHLIGHT_EDGE if edge_value > 0 else NEUTRAL_EDGE,
            "width": round(0.7 + 4.8 * normalize_metric(edge_value, 0, max_value), 3),
        })
        visual_edges.append(edge)

    graph = Graph.ByVerticesEdges(list(visual_vertices.values()), visual_edges)
    print(title)
    print("Vertices:", len(visual_vertices), "Edges:", len(visual_edges))
    return graph


def show_graph(graph, title, labels=False):
    try:
        Graph.Show(
            graph,
            vertexSizeKey="vertex_size",
            vertexColorKey="color",
            vertexLabelKey="label",
            showVertexLabel=labels,
            edgeWidthKey="width",
            edgeColorKey="color",
            backgroundColor="white",
            renderer=renderer,
        )
    except Exception as error:
        print(f"{title} graph visualization unavailable:", error)


def show_house_heatmap(metric_lookup, title, opacity=0.62, path_rooms=None):
    path_rooms = set(path_rooms or [])
    colored_cells = []
    for cell in house_cells:
        name = get_dict_value(cell, "name", "")
        info = metric_lookup.get(name, {})
        color = HIGHLIGHT_EDGE if name in path_rooms else info.get("color", "#eeeeee")
        cell = set_dictionary(cell, {
            "name": name,
            "label": name,
            "color": color,
            "metric_value": round(metric_float(info.get("value", 0)), 6),
        })
        colored_cells.append(cell)

    try:
        Topology.Show(
            colored_cells,
            faceColorKey="color",
            faceOpacity=opacity,
            backgroundColor="white",
            renderer=renderer,
        )
    except Exception as error:
        print(f"{title} house-cell heatmap unavailable:", error)


def names_in_path(row):
    return [item.strip() for item in str(row.get("path", "")).split("->") if item.strip()]


def path_edges_from_nodes(path_nodes):
    return list(zip(path_nodes[:-1], path_nodes[1:]))


def highlighted_access_graph(highlight_nodes, highlight_edge_pairs, title, highlight_rooms_on_cells=True, label_highlights=True):
    vertices = {}
    for name, base_vertex in access_vertices_by_name.items():
        is_highlight = name in highlight_nodes
        vertex = Vertex.ByCoordinates(Vertex.X(base_vertex), Vertex.Y(base_vertex), Vertex.Z(base_vertex))
        vertex = set_dictionary(vertex, {
            "name": name,
            "label": compact_label(name) if is_highlight and label_highlights else "",
            "color": HIGHLIGHT_EDGE if is_highlight else NEUTRAL_NODE,
            "vertex_size": 22 if is_highlight else 4.5,
            "type": get_dict_value(base_vertex, "type", "node"),
        })
        vertices[name] = vertex

    highlight_edge_pairs = {tuple(sorted(pair)) for pair in highlight_edge_pairs}
    edges = []
    for row in access_edge_rows:
        a, b = row["node_a"], row["node_b"]
        if a not in vertices or b not in vertices:
            continue
        is_highlight = tuple(sorted([a, b])) in highlight_edge_pairs
        edge = Edge.ByVertices([vertices[a], vertices[b]])
        edge = set_dictionary(edge, {
            "color": HIGHLIGHT_EDGE if is_highlight else NEUTRAL_EDGE,
            "width": 6 if is_highlight else 0.55,
        })
        edges.append(edge)

    graph = Graph.ByVerticesEdges(list(vertices.values()), edges)
    print(title)
    print("Highlighted nodes:", len(highlight_nodes))
    show_graph(graph, title, labels=label_highlights)

    if highlight_rooms_on_cells:
        room_nodes = [node for node in highlight_nodes if node in room_vertices_by_name]
        room_lookup = {room: {"value": 1, "color": HIGHLIGHT_EDGE, "size": 20} for room in room_nodes}
        show_house_heatmap(room_lookup, f"{title} on house cells", opacity=0.72, path_rooms=room_nodes)


def route_via_nodes(start_name, via_nodes, end_name):
    total_path = []
    total_distance = 0.0
    sequence = [start_name] + list(via_nodes) + [end_name]
    for source, target in zip(sequence[:-1], sequence[1:]):
        path, distance = topologic_shortest_path_names(access_graph, source, target, access_weighted_adjacency)
        if not path or distance == math.inf:
            return None, math.inf
        if total_path:
            total_path.extend(path[1:])
        else:
            total_path.extend(path)
        total_distance += distance
    return total_path, total_distance


def compute_street_door_paths():
    street_exit_options = [
        {"exit_name": "Door_049", "exit_label": "Principal pedestrian exit", "via": []},
        {"exit_name": "Door_014", "exit_label": "Vehicular / emergency pedestrian exit", "via": []},
        {"exit_name": "Door_048", "exit_label": "Door_048 via Door_061", "via": ["Door_061"]},
    ]
    available_options = []
    for option in street_exit_options:
        required_nodes = [option["exit_name"]] + option["via"]
        missing = [node for node in required_nodes if node not in access_vertices_by_name]
        if missing:
            print("Street-door option skipped because nodes are missing:", option["exit_label"], missing)
            continue
        available_options.append(option)

    rows = []
    room_nodes = [row["room"] for row in room_metric_rows if row["room"] in access_vertices_by_name]
    for room_name in room_nodes:
        best = None
        for option in available_options:
            path, distance = route_via_nodes(room_name, option["via"], option["exit_name"])
            if not path or distance == math.inf:
                continue
            row = {
                "start_room": room_name,
                "street_exit": option["exit_name"],
                "street_exit_label": option["exit_label"],
                "via_nodes": " -> ".join(option["via"]),
                "path_length_model_units": round(float(distance), 6),
                "hops": max(len(path) - 1, 0),
                "path": " -> ".join(path),
            }
            if best is None or row["path_length_model_units"] < best["path_length_model_units"]:
                best = row
        if best:
            rows.append(best)

    rows = sorted(rows, key=lambda row: row["path_length_model_units"], reverse=True)
    write_csv(OUTPUT_DIR / "A2_street_door_shortest_paths.csv", rows)
    print_rows(rows, limit=15)
    return rows


def community_lookup():
    communities = sorted({str(row.get("community", "")) for row in room_metric_rows if str(row.get("community", ""))})
    color_by_community = {
        community: COMMUNITY_COLORS[index % len(COMMUNITY_COLORS)]
        for index, community in enumerate(communities)
    }
    lookup = {}
    for row in room_metric_rows:
        community = str(row.get("community", ""))
        lookup[row["room"]] = {
            "value": community,
            "color": color_by_community.get(community, NEUTRAL_NODE),
            "size": 14,
        }
    return lookup


def print_metric_caption(title, metric_name):
    print("-" * 72)
    print(title)
    print(metric_name)
    print("Blue = low | Yellow/Orange = medium | Red = high")
    print("Node size follows the same metric value.")


# 1. Degree / Connectivity.
print_metric_caption("1. Connectivity Heatmap", "Degree Centrality")
degree_lookup = room_metric_lookup("degree_centrality")
degree_graph = graph_from_metric(room_vertices_by_name, room_edge_rows, degree_lookup, "Degree Centrality / Connectivity", label_policy="none")
show_graph(degree_graph, "Degree Centrality / Connectivity")
show_house_heatmap(degree_lookup, "Degree Centrality / Connectivity")

# 2. Closeness / Integration. Labels only for red/high nodes and circulation spaces.
print_metric_caption("2. Accessibility Heatmap", "Closeness Centrality / Integration proxy")
closeness_lookup = room_metric_lookup("closeness_centrality")
closeness_graph = graph_from_metric(
    room_vertices_by_name,
    room_edge_rows,
    closeness_lookup,
    "Closeness Centrality / Integration",
    label_policy="high_or_circulation",
)
show_graph(closeness_graph, "Closeness Centrality / Integration", labels=True)
show_house_heatmap(closeness_lookup, "Closeness Centrality / Integration")

# 3. Betweenness / Choice.
print_metric_caption("3. Bottleneck Heatmap", "Betweenness Centrality / Choice proxy")
betweenness_lookup = access_metric_lookup("betweenness_centrality")
betweenness_graph = graph_from_metric(access_vertices_by_name, access_edge_rows, betweenness_lookup, "Betweenness Centrality / Choice", label_policy="high")
show_graph(betweenness_graph, "Betweenness Centrality / Choice", labels=True)
room_choice_lookup = room_betweenness_lookup()
show_house_heatmap(room_choice_lookup, "Room Betweenness / Choice")

# 4. Critical shortest path. Labels only for the highlighted path nodes.
print("-" * 72)
print("4. Critical Shortest Path")
if shortest_path_rows:
    worst_path_row = shortest_path_rows[0]
    path_nodes = names_in_path(worst_path_row)
    path_edges = path_edges_from_nodes(path_nodes)
    highlighted_access_graph(
        set(path_nodes),
        path_edges,
        f"Longest proxy route: {worst_path_row['start_room']} to {worst_path_row['nearest_exit_proxy']}",
        label_highlights=True,
    )
else:
    print("No shortest path rows available to visualize.")

# 4b. Additional street-door shortest-path analysis.
print("-" * 72)
print("4b. Street Door Shortest Paths")
print("Street doors: Door_049 principal pedestrian, Door_014 vehicular/emergency pedestrian, Door_048 via Door_061.")
street_door_path_rows = compute_street_door_paths()
if street_door_path_rows:
    worst_street_row = street_door_path_rows[0]
    street_path_nodes = names_in_path(worst_street_row)
    street_path_edges = path_edges_from_nodes(street_path_nodes)
    highlighted_access_graph(
        set(street_path_nodes),
        street_path_edges,
        f"Longest street-door route: {worst_street_row['start_room']} to {worst_street_row['street_exit_label']}",
        label_highlights=True,
    )
else:
    print("No street-door path rows available to visualize.")

# 5. Bridges and cut vertices. Labels only for the highlighted nodes to avoid overlap.
print("-" * 72)
print("5. Bridges and Cut Vertices")
print("Labels are shown only for highlighted nodes to keep text near 60 percent of the previous visual density.")
bridge_nodes = set()
bridge_pairs = []
for row in bridge_rows:
    a = row.get("node_a", "")
    b = row.get("node_b", "")
    if a and b:
        bridge_nodes.update([a, b])
        bridge_pairs.append((a, b))
cut_nodes = {row.get("node", "") for row in cut_vertex_rows if row.get("node", "")}
highlighted_access_graph(
    bridge_nodes | cut_nodes,
    bridge_pairs,
    "Bridge edges and cut vertices",
    label_highlights=True,
)

# 6. Communities / Clusters.
print("-" * 72)
print("6. Community / Cluster Zones")
community_metric_lookup = community_lookup()
if any(info.get("value") not in ["", None] for info in community_metric_lookup.values()):
    community_graph = graph_from_metric(
        room_vertices_by_name,
        room_edge_rows,
        community_metric_lookup,
        "Community / Cluster View",
        neutral_color=NEUTRAL_NODE,
        label_policy="none",
    )
    show_graph(community_graph, "Community / Cluster View")
    show_house_heatmap(community_metric_lookup, "Community / Cluster Zones", opacity=0.55)
else:
    print("No community labels available to visualize.")



## 16. Visualization Convention for the Report

Use this legend to read the graph visualizations and to place beside the report in the final PDF.

<div style="max-width: 980px; font-family: Arial, sans-serif; border: 1px solid #d0d0d0; padding: 18px 20px; border-radius: 8px; background: #ffffff;">
  <div style="font-size: 20px; font-weight: 700; margin-bottom: 6px;">Graph Metric Reading Convention</div>
  <div style="font-size: 13px; color: #555; margin-bottom: 16px;">The same visual logic is used for Degree, Closeness, and Betweenness Centrality.</div>

  <svg width="920" height="245" viewBox="0 0 920 245" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Graph metric visualization legend">
    <defs>
      <linearGradient id="centralityGradient" x1="0%" y1="0%" x2="100%" y2="0%">
        <stop offset="0%" stop-color="#313695"/>
        <stop offset="14%" stop-color="#4575b4"/>
        <stop offset="28%" stop-color="#74add1"/>
        <stop offset="42%" stop-color="#abd9e9"/>
        <stop offset="56%" stop-color="#ffffbf"/>
        <stop offset="70%" stop-color="#fdae61"/>
        <stop offset="84%" stop-color="#f46d43"/>
        <stop offset="100%" stop-color="#a50026"/>
      </linearGradient>
    </defs>

    <text x="0" y="18" font-size="16" font-weight="700" fill="#222">Centrality heatmap</text>
    <rect x="0" y="34" width="540" height="28" rx="4" fill="url(#centralityGradient)" stroke="#bdbdbd"/>
    <text x="0" y="82" font-size="13" fill="#333">Low value</text>
    <text x="238" y="82" font-size="13" text-anchor="middle" fill="#333">Medium</text>
    <text x="540" y="82" font-size="13" text-anchor="end" fill="#333">High value</text>
    <text x="0" y="104" font-size="12" fill="#555">Blue spaces are less central. Yellow/orange/red spaces are more structurally important in the selected metric.</text>

    <text x="0" y="145" font-size="16" font-weight="700" fill="#222">Node size</text>
    <circle cx="32" cy="186" r="7" fill="#74add1" stroke="#222" stroke-width="0.6"/>
    <circle cx="105" cy="186" r="14" fill="#fdae61" stroke="#222" stroke-width="0.6"/>
    <circle cx="205" cy="186" r="24" fill="#d73027" stroke="#222" stroke-width="0.6"/>
    <text x="32" y="225" font-size="12" text-anchor="middle" fill="#333">Low</text>
    <text x="105" y="225" font-size="12" text-anchor="middle" fill="#333">Medium</text>
    <text x="205" y="225" font-size="12" text-anchor="middle" fill="#333">High</text>

    <text x="330" y="145" font-size="16" font-weight="700" fill="#222">Edges and routes</text>
    <line x1="330" y1="177" x2="435" y2="177" stroke="#cfcfcf" stroke-width="2"/>
    <text x="455" y="182" font-size="13" fill="#333">regular connection</text>
    <line x1="330" y1="210" x2="435" y2="210" stroke="#d73027" stroke-width="6"/>
    <text x="455" y="215" font-size="13" fill="#333">highlighted shortest path, bridge, or bottleneck</text>

    <text x="650" y="18" font-size="16" font-weight="700" fill="#222">Metric meaning</text>
    <circle cx="666" cy="50" r="7" fill="#d73027"/>
    <text x="684" y="55" font-size="13" fill="#333">Degree: many direct connections</text>
    <circle cx="666" cy="82" r="7" fill="#fdae61"/>
    <text x="684" y="87" font-size="13" fill="#333">Closeness: globally accessible space</text>
    <circle cx="666" cy="114" r="7" fill="#a50026"/>
    <text x="684" y="119" font-size="13" fill="#333">Betweenness: critical connector</text>
    <rect x="659" y="146" width="14" height="14" fill="#1f77b4"/>
    <rect x="678" y="146" width="14" height="14" fill="#ff7f0e"/>
    <rect x="697" y="146" width="14" height="14" fill="#2ca02c"/>
    <text x="724" y="158" font-size="13" fill="#333">Community: detected cluster or zone</text>
  </svg>

  <div style="font-size: 12px; color: #555; margin-top: 10px;">
    Inspired by network centrality heatmap conventions where node color represents metric intensity and edges reveal graph relationships.
  </div>
</div>


## 17. Short Report


In [ ]:
def first_or_empty(rows):
    return rows[0] if rows else {}

most_connected = first_or_empty(room_metric_rows)
most_accessible = first_or_empty(sorted(room_metric_rows, key=lambda r: r["closeness_centrality"], reverse=True))
main_bottleneck = first_or_empty(bottleneck_rows)
worst_path = first_or_empty(shortest_path_rows)
community_values = sorted({str(r.get("community", "")) for r in room_metric_rows if str(r.get("community", ""))})

report = f"""
# Assignment 2 Short Report - Graph Analysis of Aranjuez Model

## Method
The architectural model was represented as two related TopologicPy graphs. The room adjacency graph uses rooms as nodes and door-mediated room-to-room relationships as edges. The access graph uses rooms, doors, and windows as nodes; edges connect each aperture to the room or rooms detected around its centroid. Edges are treated as unweighted for centrality and as centroid-distance weighted for the shortest-path proxy.

## Key Results
- Most directly connected space: **{most_connected.get('room', 'N/A')}** with degree centrality **{most_connected.get('degree_centrality', 'N/A')}**.
- Most globally accessible space: **{most_accessible.get('room', 'N/A')}** with closeness centrality **{most_accessible.get('closeness_centrality', 'N/A')}**.
- Strongest bottleneck / connector: **{main_bottleneck.get('node', 'N/A')}** with betweenness centrality **{main_bottleneck.get('betweenness_centrality', 'N/A')}**.
- Longest proxy route to an exterior aperture starts at **{worst_path.get('start_room', 'N/A')}** and reaches **{worst_path.get('nearest_exit_proxy', 'N/A')}** in **{worst_path.get('path_length_model_units', 'N/A')}** model units.
- Detected community labels: **{', '.join(community_values) if community_values else 'not available'}**.

## Interpretation
High degree centrality identifies spaces that work as local distributors because they connect to many neighboring rooms through doors. High closeness centrality identifies spaces that are topologically near the rest of the building, which usually indicates strong accessibility and a likely role in circulation. High betweenness centrality marks rooms or apertures that many shortest routes depend on; these elements behave as critical connectors or bottlenecks.

## Spatial Organization
The graph reveals how circulation is structured by the relationship between rooms and apertures rather than by geometry alone. Central rooms and circulation spaces organize movement, while doors with high betweenness indicate thresholds where multiple routes converge. Communities suggest functional or spatial zones, such as clusters of storage rooms, circulation cores, service areas, or repeated room groups. Shortest paths to exterior aperture proxies provide an accessibility reading, not a code-compliant egress certification.

## Why Graph Analysis Is Useful
Graph analysis translates architectural geometry into relational evidence. It helps identify hierarchy, accessibility, connectivity, bottlenecks, and zones that can be hard to compare visually in a complex model. The metrics should be read together with drawings, dimensions, program, and design intent.
"""

display(Markdown(report))
(OUTPUT_DIR / "A2_short_report.md").write_text(report, encoding="utf-8")
print("Exported:", OUTPUT_DIR / "A2_short_report.md")
